# Building a Vulnerability Library

Build, persist, and incrementally update a complete NVD vulnerability library
using `CacheBuilder`.

## CacheBuilder Options

Three modes for selecting which NVD feeds to download:

| Mode | Example | Use Case |
|------|---------|----------|
| Specific years | `CacheBuilder(years=[2024, 2025])` | Targeted analysis |
| Exclude years | `CacheBuilder(exclude_years=[2002, 2003])` | Skip old data |
| All years | `CacheBuilder()` | Complete library (2002-present) |
| Modified feed | `CacheBuilder(modified_only=True)` | Quick update (~8 days) |
| Recent feed | `CacheBuilder(recent_only=True)` | New CVEs only (~8 days) |

In [ ]:
from vulnflow.helpers import CacheBuilder

# Quick subset (not executed — see "Complete Library" below for the real build)
# ds = CacheBuilder(years=[2024, 2025]).build()

# Exclude old data
# ds = CacheBuilder(exclude_years=[2002, 2003, 2004]).build()

# All years (2002-present)
# ds = CacheBuilder().build()

# Special feeds
# ds = CacheBuilder(modified_only=True).build()   # Last ~8 days
# ds = CacheBuilder(recent_only=True).build()     # Last ~8 days (new only)

print("See table above for usage — complete library built in next section.")

## Complete Library

In [ ]:
from vulnflow import VulnDataset

CACHE_PATH = "../data/complete_library.parquet"

ds = CacheBuilder().ensure(CACHE_PATH)
print(f"{len(ds):,} CVEs loaded")

In [ ]:
import matplotlib.pyplot as plt

yearly = ds.count_by("year")
print(f"{len(ds):,} CVEs  |  Year range: {int(yearly['year'].min())}-{int(yearly['year'].max())}")

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(yearly["year"], yearly["count"], color="#1f77b4", edgecolor="white")
ax.set_title("CVEs per Year", fontsize=16)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5, axis="y")
plt.tight_layout()
plt.show()

## Incremental Updates

Merge the Modified feed into an existing library to pick up recent changes.

In [ ]:
before = len(ds)
delta = CacheBuilder(modified_only=True).build()

# Save delta, then reload both files with upsert semantics (deduplicates by cve_id)
delta.to_parquet("../data/delta.parquet")
ds = VulnDataset.from_parquet([str(CACHE_PATH), "../data/delta.parquet"])
ds.to_parquet(str(CACHE_PATH))
print(f"Before: {before:,} → After: {len(ds):,} CVEs (delta: {len(delta):,})")

## Parquet Round-Trip

Verify lossless save/load.

In [ ]:
ds.to_parquet(str(CACHE_PATH))
reloaded = VulnDataset.from_parquet(str(CACHE_PATH))
assert len(reloaded) == len(ds), f"Mismatch: {len(reloaded)} != {len(ds)}"
print(f"Round-trip OK: {len(reloaded):,} CVEs, {len(reloaded.columns)} columns")